# SunSight v3 Pro (Kaggle GPU — Anti-Putus Wi-Fi!): `YOLO11m-Seg` + `mask_ratio=2` + CLAHE LAB + 25% Grayscale Anti-Color-Bias

Upgrade khusus v3 untuk mengatasi:
- **Gedung luas/panjang/kompleks** (Botani Square, IPB Baranangsiang, BTM, Lippo Plaza, RS, Unpak)
- **Atap dak beton, cokelat tua, & genteng kusam** (lewat CLAHE LAB + **25% salinan Grayscale/Desaturated** di data latih)
- **Mask tajam anti-blobby** lewat **`mask_ratio=2`** (resolusi mask internal **4× lebih padat**: `512×512` pada `imgsz=1024`!) + **`yolo11m-seg.pt`**
- **Multi-scale `152 m (2×2)` & `228 m (3×3)`** agar akurat di skala `Optimal` maupun `20 m` Studio.

Jalankan lewat **Save Version → Save & Run All (Commit)** dengan Accelerator **GPU T4 x2** atau **GPU P100** + **Internet: On**!

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime opencv-python-headless

In [ ]:
# 1. Muat dataset v3 dari /kaggle/input/ dan terapkan CLAHE LAB + 25% Grayscale/Desaturated copies pada split train
import random, shutil, yaml
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

INPUT_ROOT = Path('/kaggle/input')
WORK_DATA = Path('/kaggle/working/sunsight_v3_clahe')

yamls = list(INPUT_ROOT.rglob('dataset.yaml'))
if not yamls:
    raise FileNotFoundError('Tambahkan Kaggle Dataset sunsight-yolo11m-seg-v3-pro lewat tombol Add Input!')

SRC_DATASET = yamls[0].parent
print('Dataset v3 ditemukan di:', SRC_DATASET)

if WORK_DATA.exists():
    shutil.rmtree(WORK_DATA)
shutil.copytree(SRC_DATASET, WORK_DATA)

def apply_clahe_lab(bgr_img):
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)

# Terapkan CLAHE LAB pada seluruh split (sesuai produksi USE_CLAHE_LAB=true)
for split in ('train', 'val', 'test'):
    for img_p in tqdm(list((WORK_DATA / 'images' / split).glob('*.*')), desc=f'CLAHE {split}'):
        bgr = cv2.imread(str(img_p))
        if bgr is not None:
            cv2.imwrite(str(img_p), apply_clahe_lab(bgr), [int(cv2.IMWRITE_JPEG_QUALITY), 92])

# Tambah 25% salinan Grayscale/Desaturated HANYA di split train agar model 100% kebal bias warna
# (atap cokelat tua, dak beton abu, & asbes kusam terdeteksi berdasarkan struktur/tepi, bukan warna oranye!)
rng = random.Random(42)
train_imgs = sorted((WORK_DATA / 'images' / 'train').glob('*.jpg'))
pos_train = [p for p in train_imgs if not p.name.startswith('hnv2_')]
gray_picks = rng.sample(pos_train, k=max(1, int(len(pos_train) * 0.25)))

for img_p in tqdm(gray_picks, desc='Tambah 25% Grayscale Anti-Color-Bias (train only)'):
    bgr = cv2.imread(str(img_p))
    lbl_p = WORK_DATA / 'labels' / 'train' / f'{img_p.stem}.txt'
    if bgr is None or not lbl_p.exists():
        continue
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    # Campur 85% grayscale + 15% warna asli agar menyerupai beton/asbes/cokelat kusam alami
    desat = cv2.addWeighted(gray_bgr, 0.85, bgr, 0.15, 0)
    out_img = WORK_DATA / 'images' / 'train' / f'{img_p.stem}_desat.jpg'
    out_lbl = WORK_DATA / 'labels' / 'train' / f'{img_p.stem}_desat.txt'
    cv2.imwrite(str(out_img), desat, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    shutil.copy2(lbl_p, out_lbl)

cfg = yaml.safe_load((WORK_DATA / 'dataset.yaml').read_text(encoding='utf-8'))
cfg['path'] = str(WORK_DATA)
(WORK_DATA / 'dataset.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Total train images setelah augmentasi desaturasi:', len(list((WORK_DATA / 'images' / 'train').glob('*.jpg'))))

In [ ]:
# 2. Training YOLO11m-Seg (Medium) — Hemat VRAM (anti-OOM pada T4 15 GB!) + Multi-GPU otomatis
import json, os
import torch
from ultralytics import YOLO

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
n_gpu = torch.cuda.device_count()
train_device = list(range(n_gpu)) if n_gpu > 1 else 0
train_batch = 8 if n_gpu > 1 else 4
print(f'GPU terdeteksi: {n_gpu} | device={train_device} | batch={train_batch}')

model = YOLO('yolo11m-seg.pt')
model.train(
    data=str(WORK_DATA / 'dataset.yaml'),
    epochs=150,
    patience=30,
    imgsz=1024,
    batch=train_batch,
    device=train_device,
    optimizer='AdamW',
    lr0=0.0008,
    cos_lr=True,
    # Hemat VRAM 10x pada tile padat (250+ atap/mosaic) tanpa menurunkan ketajaman:
    # karena poligon atap tidak saling tumpang tindih, overlap_mask=True + mask_ratio=4 aman di T4 15GB!
    mask_ratio=4,
    overlap_mask=True,
    # Augmentasi kuat anti-bias warna & multi-skala:
    hsv_h=0.05,
    hsv_s=0.8,
    hsv_v=0.55,
    degrees=180.0,
    flipud=0.5,
    fliplr=0.5,
    scale=0.45,
    mosaic=0.5,
    close_mosaic=20,
    project='/kaggle/working/runs',
    name='yolo11m_seg_v3_bogor_clahe',
    exist_ok=True,
    plots=True,
)

best_pt = Path('/kaggle/working/runs/yolo11m_seg_v3_bogor_clahe/weights/best.pt')
best_model = YOLO(str(best_pt))

# Evaluasi standar COCO/Ultralytics pada Test Set bersih
m = best_model.val(data=str(WORK_DATA / 'dataset.yaml'), split='test', imgsz=1280, max_det=300, device=0)

# Ekspor ke ONNX CPU-only (imgsz=1280, simplify=True) -> langsung siap untuk FastAPI SunSight!
onnx_path = Path(best_model.export(format='onnx', imgsz=1280, simplify=True, opset=17, device=0))
shutil.copy2(onnx_path, '/kaggle/working/best_v3.onnx')
shutil.copy2(best_pt, '/kaggle/working/best_v3.pt')

summary = {
    'v3_yolo11m_seg_clahe_desat': {
        'mask_precision': round(float(m.seg.mp), 4),
        'mask_recall': round(float(m.seg.mr), 4),
        'mask_map50': round(float(m.seg.map50), 4),
        'mask_map50_95': round(float(m.seg.map), 4),
    }
}
Path('/kaggle/working/journal_metrics_v3.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('=== SELESAI v3! RINGKASAN METRIK TEST SET ===')
print(json.dumps(summary, indent=2))